# Predicta Semiconductor Test Analytics — Day 3.75 Validation Error Analysis

**Validation Dataset**: `ml/data/processed/validation.csv` (6,000 records / 12 unseen wafers)  
**Baseline Model**: `ml/models/predicta_xgboost_baseline.json`  
**Operating Threshold**: `0.35`  
**Plot Artifact**: `ml/analysis/plots/defect_recall.svg`  

> [!IMPORTANT]
> Zero model retraining or test set evaluation is performed. All diagnostic analysis is executed strictly on validation predictions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

val_df = pd.read_csv('../data/processed/validation.csv')
raw_df = pd.read_csv('../data/synthetic/predicta_dataset_v3_50000.csv')
print(f'Validation Records Loaded: {len(val_df)}')


Loaded 6,000 validation records joined with defect_type labels.
Confusion Matrix at Threshold 0.35: TP=614, TN=4586, FP=607, FN=193


--- 
## Section 1 — Defect-Wise Detection Recall
Recall breakdown across each of the 7 defect types at Threshold 0.35.

In [2]:
# Defect-wise performance summary table


Defect-Wise Detection Recall Breakdown:
  THERMAL_ANOMALY    :  93.07% Recall ( 94 / 101 caught)
  POWER_ANOMALY      :  92.16% Recall ( 94 / 102 caught)
  LOW_VOLTAGE        :  91.87% Recall (113 / 123 caught)
  HIGH_LEAKAGE       :  81.46% Recall (145 / 178 caught)
  TIMING_FAILURE     :  80.31% Recall (102 / 127 caught)
  PROCESS_VARIATION  :  58.89% Recall ( 53 /  90 caught)
  EQUIPMENT_DRIFT    :  15.12% Recall ( 13 /  86 caught)  [HARDEST DEFECT]


--- 
## Section 2 — False-Negative (FN) & False-Positive (FP) Feature Comparisons
Physical parameter comparisons between missed failures (FN) vs caught failures (TP) and false alarms (FP) vs true normals (TN).

In [3]:
# Physical parameter comparison


Feature Comparison Summary:
  - FN (Missed Defect) Leakage Current Mean: 136.87 µA vs TP Mean: 147.05 µA (Near normal baseline 132 µA)
  - FP (False Alarm) Propagation Delay Mean: 13.39 ns vs TN Mean: 12.37 ns (Elevated upper normal range)


--- 
## Section 3 — Final Validation Error Summary for ML Lead

```text
=========================================================================
PREDICTA VALIDATION ERROR ANALYSIS — FINAL ML LEAD SUMMARY
=========================================================================
1. Overall Confusion Matrix   : TP=614, TN=4586, FP=607, FN=193
2. Overall FAIL Recall        : 76.08% at Threshold 0.35
3. Easiest Defects            : THERMAL_ANOMALY (93.07%), POWER_ANOMALY (92.16%), LOW_VOLTAGE (91.87%)
4. Hardest Defects            : EQUIPMENT_DRIFT (15.12%), PROCESS_VARIATION (58.89%)
5. False-Negative Profile     : FN are mild/low-severity defects with parameters near normal limits.
6. False-Positive Profile     : FP occur when healthy components have upper-range normal delay/temp.
7. Recommended Tuning Focus   : 1. Optimize tree depth (max_depth 6-8) & min_child_weight for subtle shifts.
                                2. Feature engineering: multi-measurement ratio features (e.g. I_leak/P_dyn).
=========================================================================
```